<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
### <center> المؤلف: ألكسندر كوروتكوف، ODS Slack krotix
    
## <center> البرنامج التعليمي
### <center> مساعد آخر للتعلم الجماعي



![العنوان](https://static1.squarespace.com/static/57dc396a03596e8da9fe6b73/t/57eef283b3db2ba633355a07/1480477568336/UBC_Bands.jpg)
<center> الصورة بواسطة بريان هوكس



### ماذا يعني التعلم الجماعي؟



**التعلم الجماعي** - هذه طريقة تستخدم خوارزميات تعلم متعددة للحصول على أداء تنبؤي أفضل (أقول ذلك عادة، ولكن ليس في أي حالة) مما يمكن الحصول عليه من أي من خوارزميات التعلم المكونة وحدها.
التقنيات الأكثر شيوعًا هي:
* التعزيز
* التعبئة
* التراص
نحن نبحث في بعض الخوارزميات الوصفية لإنشاء مجموعة يمكنها تحسين أداء المقياس والحصول على سرعة أفضل للتجربة وتبسيط التعليمات البرمجية.
أرغب في عرض العديد من المكتبات للتجميع في بيثون:
* https://github.com/rasbt/mlxtend.git - مكتبة من الأدوات المفيدة لمهام علوم البيانات اليومية.
* https://github.com/flennerhag/mlens - مكتبة للتعلم الجماعي عالي الأداء.
* https://github.com/Menelau/DESlib - مكتبة تعليمية جماعية سهلة الاستخدام تركز على تنفيذ أحدث التقنيات للمصنف الديناميكي واختيار المجموعة.
سوف نفهم استخدام المكتبات المختلفة بمثال بسيط ومخطط لحدود القرار لتصور الاختلافات.



### <center> التثبيت


In [ ]:
!pip install mlxtend
!pip install mlens
!pip install deslib


قم بإعداد دفتر ملاحظاتنا لمزيد من التجارب:
* استيراد كافة المكتبات
* تحميل بيانات المثال وتقسيمها
* عمل مصنفات للمقارنة
* إنشاء وظيفة المرافق
سنستخدم مجموعة بيانات Iris كمثال.
الميزات
* طول السيبال
* عرض سيبال
* طول البتلة
* عرض البتلة
عدد العينات: 150.متغير الهدف (منفصل): {50x سيتوسا، 50x فيرسيكولور، 50x فيرجينيكا}


In [ ]:
import itertools
import warnings

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
# common libraries
import numpy as np
from deslib.dcs import MCB
from deslib.des.knora_e import KNORAE
from deslib.static import StaticSelection
from mlens.ensemble import (BlendEnsemble, SequentialEnsemble, Subsemble,
                            SuperLearner)
from mlxtend.classifier import (EnsembleVoteClassifier, StackingClassifier,
                                StackingCVClassifier)
from mlxtend.data import iris_data
from mlxtend.plotting import plot_decision_regions
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

In [ ]:
# random seed
seed = 10

# Loading example data
X, y = iris_data()
X = X[:, [0, 2]]

# split the data into training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=seed
)

In [ ]:
# Initializing several classifiers
clf1 = LogisticRegression(random_state=seed)
clf2 = RandomForestClassifier(random_state=seed)
clf3 = SVC(random_state=seed, probability=True)

In [ ]:
def compare(classifier, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test):
    # Plotting Decision Regions
    gs = gridspec.GridSpec(2, 2)
    fig = plt.figure(figsize=(10, 8))

    # Label for our classifiers
    labels = ["Logistic Regression", "Random Forest", "RBF kernel SVM", "Ensemble"]

    classifiers = [clf1, clf2, clf3, classifier]
    for clf, label, grid in zip(
        classifiers, labels, itertools.product([0, 1], repeat=2)
    ):
        clf.fit(X, y)
        ax = plt.subplot(gs[grid[0], grid[1]])
        fig = plot_decision_regions(X=X, y=y, clf=clf, legend=2)
        plt.title(label)

    plt.show()

    for clf, label in zip(classifiers, labels):
        print(label)
        print(classification_report(clf.predict(X_test), y_test))


### <center>



فئات ملكستند:
* EnsembleVoteClassifier - مساعد تصويت الأغلبية للتصنيف
* StackingClassifier - مصنف تعريفي للتعلم الجماعي للتكديس
* StackingCVClassifier - مصنف تعريفي للتعلم الجماعي للتكديس باستخدام التحقق المتبادل لإعداد المدخلات لمصنف المستوى 2 لمنع التجهيز الزائد
دعونا نكتشف كيف يمكننا استخدامه من خلال الأمثلة.



#### EnsembleVoteClassifier
للحصول على مزيد من المعلومات، انظر https://rasbt.github.io/mlxtend/user_guide/classifier/EnsembleVoteClassifier/


In [ ]:
eclf = EnsembleVoteClassifier(clfs=[clf1, clf2, clf3], weights=[2, 1, 1], voting="soft")
compare(eclf)


#### تصنيف التراص
للحصول على مزيد من المعلومات، انظر https://rasbt.github.io/mlxtend/user_guide/classifier/StackingClassifier/


In [ ]:
sclf = StackingClassifier(
    classifiers=[clf1, clf2, clf3], meta_classifier=LogisticRegression()
)
compare(sclf)


#### StackingCVClassifier
للحصول على مزيد من المعلومات، انظر https://rasbt.github.io/mlxtend/user_guide/classifier/StackingCVClassifier/


In [ ]:
scvclf = StackingCVClassifier(
    classifiers=[clf1, clf2, clf3], meta_classifier=LogisticRegression()
)
compare(scvclf)


## <center> ملينز



لدى Mlens عدة فئات مفيدة:
* SuperLearner - مجموعة التراص 
* Subsemble - خوارزمية مجموعة خاضعة للإشراف تستخدم مجموعات فرعية من البيانات الكاملة لتناسب الطبقة
* BlendEnsemble - تستخدم المجموعة الخاضعة للإشراف المتعلم التعريفي لتقدير مصفوفة التنبؤ 
* SequentialEnsemble - مجموعة تعليمية متعددة الطبقات



#### المتعلم المتميز
للحصول على مزيد من المعلومات اتبع الرابط http://ml-ensemble.com/info/start/ensembles.html#super-learner


In [ ]:
sl = SuperLearner(folds=5, random_state=seed, verbose=2)

# Build the first layer
sl.add([clf1, clf2, clf3])
# Attach the final meta-estimator
sl.add_meta(LogisticRegression())

compare(sl)


#### فرعي
للحصول على مزيد من المعلومات اتبع الرابط http://ml-ensemble.com/info/start/ensembles.html#subsemble


In [ ]:
sub = Subsemble(partitions=3, random_state=seed, verbose=2, shuffle=True)

# Build the first layer
sub.add([clf1, clf2, clf3])
sub.add_meta(SVC())

compare(sub)


#### مزيج إنسيمبل
للحصول على مزيد من المعلومات اتبع الرابط http://ml-ensemble.com/info/start/ensembles.html#blend-ensemble


In [ ]:
be = BlendEnsemble(test_size=0.7, random_state=seed, verbose=2, shuffle=True)

# Build the first layer
be.add([clf1, clf2, clf3])
be.add_meta(LogisticRegression())

compare(be)


#### مجموعة متسلسلة
للحصول على مزيد من المعلومات اتبع الرابط http://ml-ensemble.com/info/start/ensembles.html#sequential-ensemble


In [ ]:
se = SequentialEnsemble(random_state=seed, shuffle=True)

# The initial layer is a blended layer, same as a layer in the BlendEnsemble
se.add("blend", [clf1, clf2])

# The second layer is a stacked layer, same as a layer of the SuperLearner
se.add("stack", [clf1, clf3])

# The meta estimator is added as in any other ensemble
se.add_meta(SVC())

compare(se)


## <center> DESlib



يحتوي DESlib على 23 خوارزمية وتقنيات مجمعة مختلفة مقسمة إلى 3 مجموعات:
* اختيار المجموعة الديناميكية (DES)
* اختيار المصنف الديناميكي (DCS)
* طرق خط الأساس (ثابت)دعونا نجرب بعضًا منهم من مجموعات مختلفة:
* KNORAE - خوارزمية التحديد الديناميكي للمجموعة (DES) استنادًا إلى k-Nearest Oracle-Eliminate(KNORA-E)
* MCB - خوارزمية التحديد الديناميكي للمصنف (DCS) استنادًا إلى سلوك المصنف المتعدد (MCB)
* StaticSelection - الطريقة الأساسية (الثابتة) لنموذج المجموعة الذي يحدد المصنفات N ذات الأداء الأفضل
للحصول على مزيد من المعلومات اتبع الرابط https://deslib.readthedocs.io/en/latest/api.html



#### كنوري 
للحصول على مزيد من المعلومات اتبع الرابط https://deslib.readthedocs.io/en/latest/modules/des/knora_e.html


In [ ]:
kne = KNORAE([clf1, clf2, clf3])
compare(kne)


#### ام سي بي
للحصول على مزيد من المعلومات اتبع الرابط https://deslib.readthedocs.io/en/latest/modules/dcs/mcb.html


In [ ]:
mcb = MCB([clf1, clf2, clf3])
compare(mcb)


#### التحديد الثابت
للحصول على مزيد من المعلومات اتبع الرابط https://deslib.readthedocs.io/en/latest/modules/static/static_selection.html


In [ ]:
ss = StaticSelection([clf1, clf2, clf3])
compare(ss)


### <center> ملخص
    
لقد نظرنا إلى خوارزميات مختلفة ومكتبات مختلفة، والتي قد توفر الكثير من الوقت عندما تحتاج إلى استخدام تقنية المجموعة. تعد نمذجة المجموعة طريقة فعالة لتحسين أداء نماذج التعلم الآلي لديك. إذا كنت ترغب في أن تكون على رأس قائمة المتصدرين في أي مسابقة للتعلم الآلي أو ترغب في تحسين النماذج التي تعمل عليها - فإن المجموعة هي الحل الأمثل.
    
ملاحظة: ليس هناك حل سحري... جرب أدوات وخوارزميات مختلفة.



#### الموارد: 
* https://en.wikipedia.org/wiki/Ensemble_learning
* https://rasbt.github.io/mlxtend/USER_GUIDE_INDEX/
* https://www.dataquest.io/blog/introduction-to-ensembles/
* https://www.slideshare.net/SessionsEvents/erin-ledell-machine-learning-scientist-h2oai-at-mlconf-atl-2016
* https://medium.com/@rrfd/boosting-bagging-and-stacking-ensemble-methods-with-sklearn-and-mlens-a455c0c982de
* https://www.analyticsvidhya.com/blog/2018/06/comprehensive-guide-for-ensemble-models/